In [0]:
%run ./config

In [0]:
df=spark.read.option('inferSchema', True).csv(bronze+'/myntra.csv', header=True)

In [0]:
df.display()

In [0]:
df=spark.read.option('inferSchema', True).option('header', True).csv(bronze+'/myntra.csv')

In [0]:
df=df.withColumn('date', to_date(col("order_created_date"),'yyyyMMdd'))

In [0]:
df= df.withColumn('price', round(col("mrp_revenue")/col('sales'),2))\
    .withColumn('discount', round(col("vendor_disc")/col('sales'),2))\
    .withColumnRenamed('mrp_revenue', 'total_revenue')\
    .withColumnRenamed('sales', 'total_units')\
    .withColumn('sku_name', lower(trim('style_name')))\
    .withColumn("sku_id", df["style_id"].cast(StringType()))


In [0]:
df=df.select("date", 
             "sku_name", 
             "sku_id", 
             "total_units", 
             'price',
             'total_revenue')
df=df.na.drop()

In [0]:
df.write.mode('overwrite').option('header', True).csv(silver+'/myntra.csv')

In [0]:

df=spark.read.csv(silver+'/myntra.csv', header=True, inferSchema=True)
df=df.select('sku_name').distinct()
df.display()